In [1]:
import numpy as np
import pandas as pd
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.utils import to_categorical

In [2]:
import os
os.listdir("train")


['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']

In [3]:
import os
os.listdir("test")

['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']

In [4]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout


In [5]:
img_size = 48
batch_size = 64

train_gen = ImageDataGenerator(rescale=1./255)
test_gen  = ImageDataGenerator(rescale=1./255)

train_data = train_gen.flow_from_directory(
    "train",
    target_size=(img_size, img_size),
    color_mode="grayscale",
    batch_size=batch_size,
    class_mode="categorical"
)

test_data = test_gen.flow_from_directory(
    "test",
    target_size=(img_size, img_size),
    color_mode="grayscale",
    batch_size=batch_size,
    class_mode="categorical"
)


Found 28709 images belonging to 7 classes.
Found 7178 images belonging to 7 classes.


In [6]:
model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(48,48,1)),
    MaxPooling2D(2,2),

    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),

    Dense(7, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)


c:\Users\MAAZ ASHARF\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [7]:
model.fit(
    train_data,
    epochs=5,
    validation_data=test_data
)


Epoch 1/5
449/449 ━━━━━━━━━━━━━━━━━━━━ 79s 172ms/step - accuracy: 0.3269 - loss: 1.6913 - val_accuracy: 0.4168 - val_loss: 1.5196
Epoch 2/5
449/449 ━━━━━━━━━━━━━━━━━━━━ 48s 108ms/step - accuracy: 0.4212 - loss: 1.5025 - val_accuracy: 0.4590 - val_loss: 1.4162
Epoch 3/5
449/449 ━━━━━━━━━━━━━━━━━━━━ 24s 53ms/step - accuracy: 0.4547 - loss: 1.4200 - val_accuracy: 0.4843 - val_loss: 1.3486
Epoch 4/5
449/449 ━━━━━━━━━━━━━━━━━━━━ 23s 51ms/step - accuracy: 0.4785 - loss: 1.3590 - val_accuracy: 0.4900 - val_loss: 1.3040
Epoch 5/5
449/449 ━━━━━━━━━━━━━━━━━━━━ 23s 50ms/step - accuracy: 0.4980 - loss: 1.3124 - val_accuracy: 0.5008 - val_loss: 1.2718


In [8]:
model.save("emotion_model.h5")
print("emotion_model.h5 saved successfully")


emotion_model.h5 saved successfully


In [9]:
import cv2
import numpy as np
from tensorflow.keras.models import load_model

# Load face detector
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

# Load trained emotion model
emotion_model = load_model("emotion_model.h5")

# Emotion labels (must match training order)
emotion_labels = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']

# Start laptop webcam (0 = built-in camera)
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Error: Could not open camera")
    exit()

while True:
    ret, frame = cap.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.3, 5)

    for (x, y, w, h) in faces:
        face = gray[y:y+h, x:x+w]
        face = cv2.resize(face, (48, 48))
        face = face / 255.0
        face = face.reshape(1, 48, 48, 1)

        preds = emotion_model.predict(face, verbose=0)
        emotion = emotion_labels[np.argmax(preds)]

        # Map to required emotions
        if emotion == "happy":
            final_emotion = "Happy"
        elif emotion == "sad":
            final_emotion = "Sad"
        elif emotion == "angry":
            final_emotion = "Angry"
        elif emotion == "romantic":
            final_emotion == "Romantic"
        else:
            final_emotion = "neutral"

        # Draw bounding box & emotion
        cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 255, 0), 2)
        cv2.putText(
            frame,
            final_emotion,
            (x, y-10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.9,
            (0, 255, 0),
            2
        )

    cv2.imshow("Webcam Emotion Detection", frame)

    # Press 'q' to quit
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
